# Notebook 2 - Verificação de normalidade e normalização

Este notebook parte da base limpa gerada pelo notebook 1 e cobre duas etapas encadeadas:

1. Verificação de normalidade da variável quantitativa Temperature, correspondente aos itens (c) e (e) do Artefato 5: teste de Shapiro-Wilk no conjunto completo e por região térmica, e comparação entre média e mediana.

2. Normalização e codificação: padronização da variável numérica, One-Hot e Label Encoding das variáveis categóricas e separação entre features (X) e variável-alvo (y).

Pré-requisito: executar leitura.ipynb antes deste notebook.

## Setup do ambiente

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S1_S2"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
import os
import pandas as pd
from scipy import stats

pd.set_option("display.max_columns", None)

DIR_SAIDA = "dados_intermediarios"
CAMINHO_LIMPO = os.path.join(DIR_SAIDA, "df_limpo.csv")

if not os.path.exists(CAMINHO_LIMPO):
    raise FileNotFoundError(
        "df_limpo.csv não encontrado. Execute o notebook 01_leitura.ipynb antes deste."
    )

df_limpo = pd.read_csv(CAMINHO_LIMPO)
df_limpo["Possui Active Limit"] = df_limpo["Possui Active Limit"].astype(bool)

# Nível de significância adotado em toda a subseção
ALPHA = 0.05

print("Base carregada:", df_limpo.shape)

## Seleção das variáveis

A única variável quantitativa da base é a Temperature. 

O plano original previa testar também o Active Limit, mas essa coluna é um limite tabelado, definido por norma técnica e fixo por componente, e não uma medição contínua, de modo que média e desvio padrão não têm significado sobre ela. Validada essa constatação junto ao professor de matemática, a análise foi replanejada: a Temperature passa a ser testada tanto no conjunto completo quanto segmentada pelas sete regiões térmicas da coluna Key Area, o que fornece as sete distribuições exigidas pelo artefato sem que uma variável inadequada seja forçada ao teste.

In [ ]:
var_temperatura = df_limpo["Temperature"].dropna()

variaveis = {
    "Temperature": var_temperatura,
}

for nome, serie in variaveis.items():
    print(f"{nome}: n={len(serie)}")

In [ ]:
# Confirmação de que as variáveis estão prontas para os testes
for nome, serie in variaveis.items():
    assert serie.dtype.kind in "fi", f"{nome} não é numérica"
    assert serie.isna().sum() == 0, f"{nome} ainda tem nulos"
    assert len(serie) >= 8, f"{nome} tem amostra pequena demais para Shapiro-Wilk"

print("Ambiente pronto")

## Teste de normalidade

Teste utilizado: Shapiro-Wilk

Hipóteses estatísticas do teste, que não se confundem com as hipóteses de negócio do notebook 4:
- H₀: a amostra provém de uma população com distribuição normal
- H₁: a amostra não provém de uma população com distribuição normal
- α: 0,05

Regra de decisão: rejeita-se H₀ se p-valor < α.

In [ ]:
# Limite acima do qual o Shapiro-Wilk passa a operar sobre uma amostra
LIMITE_SHAPIRO = 5000
SEED = 42


def testar_normalidade(nome, serie, alpha=ALPHA):
    """Aplica Shapiro-Wilk e devolve o resultado formatado."""
    amostra = serie
    houve_amostragem = False

    if len(serie) > LIMITE_SHAPIRO:
        amostra = serie.sample(LIMITE_SHAPIRO, random_state=SEED)
        houve_amostragem = True

    estatistica, p_valor = stats.shapiro(amostra)
    rejeita = p_valor < alpha

    return {
        "Variável": nome,
        "n": len(amostra),
        "Amostragem": houve_amostragem,
        "Estatística W": estatistica,
        "p-valor": p_valor,
        "Decisão": "Rejeita H0" if rejeita else "Não rejeita H0",
        "Conclusão": "Não segue distribuição normal" if rejeita else "Compatível com distribuição normal",
    }


# Acumulador de resultados
resultados_normalidade = []

In [ ]:
resultado_temperatura = testar_normalidade("Temperature", var_temperatura)
resultados_normalidade.append(resultado_temperatura)

print("Variável:", resultado_temperatura["Variável"])
print("n:", resultado_temperatura["n"])
print("Amostragem aplicada:", resultado_temperatura["Amostragem"])
print(f"Estatística W: {resultado_temperatura['Estatística W']:.4f}")
print(f"p-valor: {resultado_temperatura['p-valor']:.6e}")
print(f"Nível de significância (alpha): {ALPHA}")
print("Decisão:", resultado_temperatura["Decisão"])
print("Conclusão:", resultado_temperatura["Conclusão"])

### Tabela consolidada do teste de normalidade

In [ ]:
tabela_normalidade = pd.DataFrame(resultados_normalidade)

tabela_normalidade.insert(1, "Teste", "Shapiro-Wilk")
tabela_normalidade["Alpha"] = ALPHA

tabela_consolidada = tabela_normalidade[
    ["Variável", "Teste", "n", "Estatística W", "p-valor", "Alpha", "Decisão", "Conclusão"]
].copy()

tabela_consolidada["Estatística W"] = tabela_consolidada["Estatística W"].map(lambda x: f"{x:.4f}")
tabela_consolidada["p-valor"] = tabela_consolidada["p-valor"].map(lambda x: f"{x:.2e}")

print("TABELA CONSOLIDADA - TESTES DE NORMALIDADE\n")
print(tabela_consolidada.to_string(index=False))

In [ ]:
def df_para_markdown(dataframe):
    """Converte um dataframe em tabela markdown, para colagem na documentação."""
    colunas = list(dataframe.columns)
    linhas = ["| " + " | ".join(colunas) + " |",
              "|" + "|".join(["---"] * len(colunas)) + "|"]
    for _, linha in dataframe.iterrows():
        linhas.append("| " + " | ".join(str(v) for v in linha) + " |")
    return "\n".join(linhas)


print(df_para_markdown(tabela_consolidada))

## Regiões térmicas

A coluna Key Area identifica a região do fogão à qual cada termopar pertence. A base contém exatamente sete regiões, que definem a segmentação usada tanto no teste por região quanto nos histogramas do notebook 3.

A lista de regiões e os rótulos legíveis são gravados em disco junto com as estatísticas, de modo que o notebook 3 utilize exatamente a mesma segmentação e a mesma ordem de exibição.

In [ ]:
regioes = df_limpo["Key Area"].dropna().unique().tolist()

print(f"Regiões encontradas: {len(regioes)}\n")
for regiao in regioes:
    sub = df_limpo[df_limpo["Key Area"] == regiao]
    limites = sorted(sub["Active Limit"].dropna().unique())
    print(f"  {regiao:<15} | n = {len(sub):>4} | termopares: {sub['TC Name'].nunique():>2} | "
          f"limites distintos: {len(limites)}")

In [ ]:
# Ordem de exibição e rótulos legíveis das regiões
REGIOES = [
    "Mainback",
    "Coil Temps",
    "Main Vent Sys",
    "IPC Cooling",
    "IPC Comp",
    "Oven HMI",
    "CT Knobs",
]

ROTULOS_REGIAO = {
    "Mainback": "Mainback - peças da parede traseira",
    "Coil Temps": "Coil Temps - bobinas do cooktop",
    "Main Vent Sys": "Main Vent Sys - ventilação principal",
    "IPC Cooling": "IPC Cooling - ventilação cooktop/IPC",
    "IPC Comp": "IPC Comp - componentes da placa IPC",
    "Oven HMI": "Oven HMI - interface do forno",
    "CT Knobs": "CT Knobs - encoders do cooktop",
}

# Nenhuma região da base ficou de fora do recorte
faltantes = [r for r in regioes if r not in REGIOES]
assert not faltantes, f"Regiões presentes na base e ausentes da lista: {faltantes}"

print(f"Regiões selecionadas: {len(REGIOES)}")

In [ ]:
# Estatísticas descritivas e Shapiro-Wilk por região
resumo_regioes = []

for regiao in REGIOES:
    sub = df_limpo[df_limpo["Key Area"] == regiao]
    serie = sub["Temperature"].dropna()
    limites = sorted(sub["Active Limit"].dropna().unique())
    estatistica_w, p_valor = stats.shapiro(serie)

    resumo_regioes.append({
        "Região": regiao,
        "n": len(serie),
        "Termopares": sub["TC Name"].nunique(),
        "Limites": len(limites),
        "Média": round(serie.mean(), 2),
        "Mediana": round(serie.median(), 2),
        "Desvio-padrão": round(serie.std(), 2),
        "Assimetria": round(serie.skew(), 3),
        "W": round(estatistica_w, 4),
        "p-valor": f"{p_valor:.2e}",
        "Decisão": "Rejeita H0" if p_valor < ALPHA else "Não rejeita H0",
    })

tabela_regioes = pd.DataFrame(resumo_regioes)

print("ESTATÍSTICAS POR REGIÃO TÉRMICA\n")
print(tabela_regioes.to_string(index=False))

## Comparação entre média e mediana

A comparação é feita no conjunto completo da variável Temperature e, em seguida, por região térmica, mantendo a mesma segmentação utilizada nos histogramas.

Como a diferença é interpretada: em uma distribuição normal, média e mediana coincidem. O afastamento entre as duas é, portanto, um indício de assimetria. Dessa forma, média acima da mediana indica cauda alongada à direita (assimetria positiva); por outro lado, média abaixo da mediana indica cauda alongada à esquerda (assimetria negativa).

In [ ]:
media_geral = var_temperatura.mean()
mediana_geral = var_temperatura.median()
desvio_geral = var_temperatura.std()

diferenca_geral = media_geral - mediana_geral
diferenca_relativa_geral = 100 * diferenca_geral / mediana_geral
pearson_geral = 3 * diferenca_geral / desvio_geral

print("Temperature (conjunto completo)\n")
print(f"n................................: {len(var_temperatura)}")
print(f"Média............................: {media_geral:.2f} °C")
print(f"Mediana..........................: {mediana_geral:.2f} °C")
print(f"Desvio-padrão....................: {desvio_geral:.2f} °C")
print(f"Diferença (média - mediana)......: {diferenca_geral:+.2f} °C")
print(f"Diferença relativa...............: {diferenca_relativa_geral:+.2f}% da mediana")
print(f"Assimetria de Pearson............: {pearson_geral:+.3f}")
print(f"Sentido da cauda.................: {'direita' if diferenca_geral > 0 else 'esquerda'}")

In [ ]:
def classificar_sentido(diferenca, tolerancia=0.05):
    """Indica o lado da cauda sugerido pela diferença entre média e mediana."""
    if abs(diferenca) < tolerancia:
        return "Simétrica"
    return "Cauda à direita" if diferenca > 0 else "Cauda à esquerda"


def classificar_magnitude(pearson):
    """Classifica a intensidade da assimetria pelo coeficiente de Pearson."""
    intensidade = abs(pearson)
    if intensidade < 0.20:
        return "Desprezível"
    if intensidade < 0.50:
        return "Moderada"
    return "Acentuada"


print("Funções de classificação definidas.")

In [ ]:
comparacao_medidas = []

for regiao in REGIOES:
    serie = df_limpo.loc[df_limpo["Key Area"] == regiao, "Temperature"].dropna()

    media = serie.mean()
    mediana = serie.median()
    desvio = serie.std()
    diferenca = media - mediana
    diferenca_relativa = 100 * diferenca / mediana
    pearson = 3 * diferenca / desvio

    # Resultado do teste de normalidade da mesma região
    estatistica_w, p_valor = stats.shapiro(serie)

    comparacao_medidas.append({
        "Região": regiao,
        "n": len(serie),
        "Média (°C)": round(media, 2),
        "Mediana (°C)": round(mediana, 2),
        "Dif. (°C)": round(diferenca, 2),
        "Dif. (%)": round(diferenca_relativa, 2),
        "Pearson": round(pearson, 3),
        "Sentido": classificar_sentido(diferenca),
        "Magnitude": classificar_magnitude(pearson),
        "Assimetria": round(serie.skew(), 3),
        "p-valor": f"{p_valor:.2e}",
        "Decisão": "Rejeita H0" if p_valor < ALPHA else "Não rejeita H0",
    })

tabela_medidas = pd.DataFrame(comparacao_medidas)

print("COMPARAÇÃO MÉDIA x MEDIANA POR REGIÃO TÉRMICA\n")
print(tabela_medidas.to_string(index=False))

In [ ]:
# Verificação de coerência entre a diferença média-mediana e o sinal da assimetria
print("COERÊNCIA ENTRE DIFERENÇA MÉDIA-MEDIANA E ASSIMETRIA\n")

for linha in comparacao_medidas:
    mesmo_sinal = (linha["Dif. (°C)"] >= 0) == (linha["Assimetria"] >= 0)
    marcador = "OK" if mesmo_sinal else "DIVERGE"
    print(f"  {linha['Região']:<15} | dif: {linha['Dif. (°C)']:>+7.2f} °C | "
          f"assimetria: {linha['Assimetria']:>+6.3f} | {marcador}")

print("\nRegiões com divergência de sinal exigem inspeção do histograma correspondente.")

In [ ]:
colunas_artefato = ["Região", "n", "Média (°C)", "Mediana (°C)", "Dif. (°C)",
                    "Dif. (%)", "Pearson", "Sentido", "Magnitude", "p-valor", "Decisão"]

print(df_para_markdown(tabela_medidas[colunas_artefato]))

### Comentário

No conjunto completo da variável Temperature, a média supera a mediana em cerca de 7%. O deslocamento positivo indica cauda alongada à direita e reforça a rejeição de H₀ obtida no teste de Shapiro-Wilk.

Por região, porém, esse reforço não é uniforme. Em Mainback, CT Knobs e Main Vent Sys o afastamento entre as medidas é grande o bastante para sinalizar assimetria por si só, inclusive no sentido inverso em Main Vent Sys, onde a média fica abaixo da mediana e aponta corretamente a cauda à esquerda. Em Coil Temps e IPC Comp a diferença não passa de 2,5%, o que não contradiz o teste, mas também não o reforça. Já em IPC Cooling e Oven HMI as duas medidas praticamente coincidem apesar da rejeição inequívoca de H₀, pois a não normalidade ali vem da bimodalidade e do formato das caudas, e não de deslocamento do centro.

A comparação entre média e mediana, portanto, reforça o teste no conjunto completo e em três das sete regiões, é inconclusiva em duas e falha em detectá-la em outras duas. Isso mostra que os três instrumentos são complementares: as medidas de posição detectam apenas assimetria, o histograma revela a forma completa da distribuição, incluindo multimodalidade, e o teste fornece a decisão formal, ainda que sensível ao tamanho da amostra.

## Medidas de referência para o escalonamento

Antes que a padronização seja aplicada, são levantadas as medidas que servem de base aos dois métodos de escalonamento considerados na documentação: o valor mínimo e o valor máximo fundamentam a normalização Min-Max, ao passo que a média e o desvio padrão fundamentam a padronização por escore-z, efetivamente adotada na célula seguinte.

O desvio padrão reportado é o populacional, obtido com ddof=0, em coerência com a fórmula empregada pelo StandardScaler. Cabe registrar que esse parâmetro difere do padrão do pandas, que utiliza ddof=1, o que explica a pequena divergência em relação ao desvio informado na tabela por região da subseção anterior.

In [ ]:
def medidas(serie):
    """Medidas de referência para o escalonamento, com desvio populacional."""
    return {
        "Count": len(serie),
        "Média": round(serie.mean(), 2),
        "Desvio Padrão (populacional)": round(serie.std(ddof=0), 2),
        "Mínimo": round(serie.min(), 2),
        "Máximo": round(serie.max(), 2),
    }


serie_completa = df_limpo["Temperature"]

tabela_variavel = pd.DataFrame([{"Variável": "Temperatura", **medidas(serie_completa)}])
tabela_escalonamento = pd.DataFrame(
    [
        {"Região": regiao, **medidas(df_limpo.loc[df_limpo["Key Area"] == regiao, "Temperature"])}
        for regiao in REGIOES
    ]
)

# As regiões particionam a base: nenhuma leitura fica de fora nem é contada duas vezes
assert tabela_escalonamento["Count"].sum() == len(serie_completa)

print(tabela_variavel.to_string(index=False))
print()
print(tabela_escalonamento.to_string(index=False))

In [ ]:
print(df_para_markdown(tabela_variavel))
print()
print(df_para_markdown(tabela_escalonamento))

## Normalização das colunas numéricas

A normalização é realizada com a biblioteca sklearn, por meio do método StandardScaler, pelo qual cada valor é convertido no seu desvio em relação à média da própria coluna, resultando em média 0 e desvio padrão 1.

O Active Limit não é normalizado. Por ser tratado como categoria, seus valores distintos são convertidos em indicadores binários na etapa de codificação, e a padronização por média e desvio padrão deixa de ser aplicável. Cabe registrar que, com essa decisão, a ordem entre os limites deixa de ser representada: sob o One-Hot Encoding, um limite de 130 °C passa a ser um rótulo sem qualquer relação com um de 125 °C ou de 136 °C, e a generalização para um sensor novo, com limite intermediário não observado no treino, não será possível.

A Margem de Seguranca também não é normalizada: por ser coluna de análise, e não feature do modelo, ela se mostra mais útil em graus Celsius, escala na qual o valor pode ser lido diretamente como quantos graus faltam até o limite.

In [ ]:
%pip install scikit-learn

In [ ]:
from sklearn.preprocessing import StandardScaler

# Active Limit passou a ser categórico e é codificado adiante, não normalizado.
# Margem de Seguranca fica de fora: é coluna de análise (em °C), não feature.
colunas_para_normalizar = ["Temperature"]

# reset_index renumera as linhas em 0, 1, 2, ... A numeração herdada de
# df_limpo tinha lacunas, deixadas pelo dropna da Temperature, que preserva
# os índices originais de longo_df. A renumeração se propaga para df_codif,
# X e y, que derivam deste dataframe.
df_normal = df_limpo.reset_index(drop=True).copy()

scaler = StandardScaler()
df_normal[colunas_para_normalizar] = scaler.fit_transform(df_normal[colunas_para_normalizar])

print("Média após padronização:", round(df_normal["Temperature"].mean(), 6))
print("Desvio-padrão após padronização:", round(df_normal["Temperature"].std(), 6))
df_normal.head()

## Codificação das colunas categóricas

Para as colunas categóricas, dois métodos foram utilizados. O primeiro é o One-Hot Encoding, aplicado às colunas de baixa cardinalidade: por ele, é criada uma coluna booleana para cada valor possível, de modo que Housing, que só pode ser aberto ou fechado, dá origem a Housing_Open e Housing_Closed, sem que seja inventada uma ordem entre categorias que não a possuem. Já para TC Name, que apresenta dezenas de valores distintos, o One-Hot geraria colunas em excesso; nesse caso foi utilizado o Label Encoding, pelo qual é atribuído um número inteiro único a cada sensor.

O Active Limit integra o grupo tratado por One-Hot Encoding, em coerência com sua classificação como variável categórica. Seus valores distintos dão origem a um indicador binário cada, entre os quais o correspondente ao valor 0, que identifica os sensores sem limite de segurança aplicável.

Em consequência, a coluna auxiliar Possui Active Limit torna-se perfeitamente redundante nesta etapa, uma vez que passa a ser o complemento exato do indicador Active Limit_0.0. Ela é mantida em df_limpo, onde continua a ser utilizada tanto para leitura da análise quanto para mascarar a Margem de Segurança nos casos não aplicáveis, mas é retirada do conjunto de features na separação entre X e y.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Baixa cardinalidade = One-Hot Encoding
colunas_onehot = [
    "Active Limit", "Key Area", "Reason", "Blower Direction", "Housing",
    "Board Position", "Run Case", "Losses",
    "Deflector Hole", "Material Coil Support"
]
df_codif = pd.get_dummies(df_normal, columns=colunas_onehot)

# Alta cardinalidade = Label Encoding
le = LabelEncoder()
df_codif["TC Name"] = le.fit_transform(df_codif["TC Name"])

print("Colunas após a codificação:", df_codif.shape[1])
print("Sensores codificados:", len(le.classes_))
df_codif.head()

## Separação entre features (X) e variável-alvo (y)

O modelo preditivo tem como alvo a Temperature: dada uma configuração de ensaio e um sensor, deve ser prevista a temperatura que aquele componente atingirá. Isso define o que pode e o que não pode ser incluído em X:

- Temperature é retirada das features por ser a própria variável-alvo, y.

- Margem de Seguranca é retirada por vazamento de dados, uma vez que é calculada como Active Limit menos Temperature e, portanto, carrega a resposta em si. Ela permanece disponível em df_limpo para a análise descritiva.

- TC Number é retirado por ser apenas um identificador sequencial, redundante em relação ao TC Name já codificado.

- Run é retirado por identificar o ensaio específico, e não uma característica física da configuração; sua manutenção levaria o modelo a memorizar o experimento em vez de aprender o comportamento térmico.

- Possui Active Limit é retirada por redundância, uma vez que, após a codificação, passa a ser o complemento exato do indicador Active Limit_0.0. Duas colunas idênticas configuram colinearidade perfeita, situação inofensiva para modelos baseados em árvores, mas problemática para regressão linear e para modelos com regularização, nos quais o peso atribuído é dividido entre elas e a interpretação dos coeficientes fica distorcida.

- Active Limit permanece, agora sob a forma dos indicadores binários gerados pelo One-Hot Encoding, por ser uma especificação do componente conhecida antes do ensaio e, portanto, uma feature legítima.

In [ ]:
# Variável-alvo: a temperatura que o modelo deve prever.
y = df_codif["Temperature"]

# Features: tudo menos o alvo, a margem (vazamento), os identificadores de
# sensor e de ensaio, e a flag Possui Active Limit, que após a codificação
# passou a ser idêntica ao indicador Active Limit_0.0.
colunas_para_excluir = ["TC Number", "Run", "Temperature",
                        "Margem de Seguranca", "Possui Active Limit"]
X = df_codif.drop(columns=colunas_para_excluir)

print("X:", X.shape, "| y:", y.shape)
print("Alguma coluna de texto sobrando em X?", X.select_dtypes(include="object").columns.tolist())
X.head()

## Exportação

In [ ]:
df_normal.to_csv(os.path.join(DIR_SAIDA, "df_normal.csv"), index=False)
X.to_csv(os.path.join(DIR_SAIDA, "X.csv"), index=False)
y.to_frame("Temperature").to_csv(os.path.join(DIR_SAIDA, "y.csv"), index=False)
tabela_regioes.to_csv(os.path.join(DIR_SAIDA, "tabela_regioes.csv"), index=False)
tabela_medidas.to_csv(os.path.join(DIR_SAIDA, "tabela_medidas.csv"), index=False)

print("Arquivos gravados em", DIR_SAIDA)
for arquivo in sorted(os.listdir(DIR_SAIDA)):
    print(" -", arquivo)

tabela_escalonamento.to_csv(os.path.join(DIR_SAIDA, "tabela_escalonamento.csv"), index=False)